# 00 — Live 311 Data Ingestion

Fetches new noise complaint records from the NYC Open Data API and appends them to the existing cleaned parquet. Run this before re-running notebooks 01, 02, and 03
to retrain on fresh data.

Runs every month with github actions-  automatically pulls fresh data from the NYC Open Data API every month.
**Data source:** [311 Service Requests from 2020 to Present](https://data.cityofnewyork.us/Social-Services/311-Service-Requests-from-2020-to-Present/erm2-nwe9/about_data)  
**API endpoint:** `https://data.cityofnewyork.us/resource/erm2-nwe9.json`  
**No API key required** — NYC Open Data is public.

**How it works:**
- `last_ingested.txt` stores the timestamp of the most recent record we've pulled
- Each run fetches only records created after that timestamp (incremental)
- New records are cleaned to match your existing schema and appended
- On first run (no existing parquet), fetches everything from 2020-01-01

## Imports

In [2]:
import requests
import pandas as pd
import numpy as np
import os
import time
from pathlib import Path
from datetime import datetime, timezone

# Paths
CLEANED_PARQUET   = "../data/processed/noise_cleaned.parquet"
LAST_INGESTED_FILE = "../data/processed/last_ingested.txt"
NEIGHBORHOODS_CSV  = "../data/raw/neighborhoods.csv"

# API config
API_ENDPOINT = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"
PAGE_SIZE    = 50_000   # Socrata max per request

NOISE_TYPES = [
    'Noise - Residential',
    'Noise - Commercial',
    'Noise - Street/Sidewalk',
    'Noise - Vehicle',
    'Noise - Park',
    'Noise - Helicopter',
    'Noise - House of Worship',
    'Noise'
]

print("Setup complete.")

Setup complete.


## Step 1: Determine Fetch Window

If `last_ingested.txt` exists, fetch from that date forward.
If not, that means we need to do the first run, so we fetch from 2020-01-01.

In [3]:
if Path(LAST_INGESTED_FILE).exists():
    with open(LAST_INGESTED_FILE, 'r') as f:
        fetch_from = f.read().strip()
    print(f"Incremental fetch from: {fetch_from}")
else:
    fetch_from = "2020-01-01T00:00:00"
    print(f"First run — fetching from: {fetch_from}")

fetch_started_at = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S")
print(f"Fetch started at: {fetch_started_at} UTC")

Incremental fetch from: 2026-05-31T02:06:18
Fetch started at: 2026-06-02T19:39:11 UTC


## Step 2: Fetch from NYC 311 / Socrata API 

NYC 311 Data uses Socrata as it's API. limits each response to 50,000 rows. We paginate with `$offset`
until we get an empty page, then stop.

In [4]:
def fetch_page(offset, fetch_from, noise_types, page_size=50_000):
    """Fetch one page of 311 noise complaints from the Socrata API."""

    types_str = ", ".join(f"'{t}'" for t in noise_types)
    where = (
        f"created_date > '{fetch_from}' "
        f"AND complaint_type in({types_str})"
    )

    params = {
        "$where":  where,
        "$limit":  page_size,
        "$offset": offset,
        "$order":  "created_date ASC",
        # Only pull columns we actually use — faster and smaller
        "$select": (
            "unique_key, created_date, complaint_type, "
            "borough, incident_zip, latitude, longitude"
        )
    }

    resp = requests.get(API_ENDPOINT, params=params, timeout=60)
    resp.raise_for_status()
    return resp.json()


all_pages = []
offset    = 0
page_num  = 1

while True:
    print(f"  Fetching page {page_num} (offset {offset:,})...", end=" ")
    page = fetch_page(offset, fetch_from, NOISE_TYPES, PAGE_SIZE)

    if not page:
        print("empty — done.")
        break

    all_pages.append(pd.DataFrame(page))
    print(f"{len(page):,} rows fetched")

    if len(page) < PAGE_SIZE:
        # Last page — no need to request another
        break

    offset   += PAGE_SIZE
    page_num += 1
    time.sleep(0.5)  

if not all_pages:
    print("No new records found. Dataset is up to date.")
    new_raw = pd.DataFrame()
else:
    new_raw = pd.concat(all_pages, ignore_index=True)
    print(f"\nTotal new records fetched: {len(new_raw):,}")

  Fetching page 1 (offset 0)... 4,197 rows fetched

Total new records fetched: 4,197


## Step 3: Clean New Records

In [5]:
if new_raw.empty:
    print("No new data to clean — skipping.")
else:
    df_new = new_raw.copy()

    # Rename API columns to match schema
    df_new = df_new.rename(columns={
        'unique_key':     'Unique Key',
        'created_date':   'Created Date',
        'complaint_type': 'Complaint Type',
        'borough':        'Borough',
        'incident_zip':   'Incident Zip',
        'latitude':       'Latitude',
        'longitude':      'Longitude'
    })

    # Datetime parsing and Temporal Features
    df_new['Created Date'] = pd.to_datetime(df_new['Created Date'])
    df_new['Year']         = df_new['Created Date'].dt.year
    df_new['Month']        = df_new['Created Date'].dt.month
    df_new['Week']         = df_new['Created Date'].dt.isocalendar().week
    df_new['Day_of_Week']  = df_new['Created Date'].dt.dayofweek
    df_new['Day_Name']     = df_new['Created Date'].dt.day_name()
    df_new['Hour']         = df_new['Created Date'].dt.hour
    df_new['Date']         = df_new['Created Date'].dt.date

    # Create Time bucket
    def get_time_bucket(hour):
        if 6 <= hour < 12:   return 'morning'
        elif 12 <= hour < 18: return 'afternoon'
        elif 18 <= hour < 22: return 'evening'
        elif hour >= 22 or hour < 2: return 'night'
        else: return 'overnight'

    df_new['Time_Bucket'] = df_new['Hour'].apply(get_time_bucket)

    # Season feature collection
    df_new['Season'] = df_new['Month'].map({
        12: 'Winter', 1: 'Winter', 2: 'Winter',
         3: 'Spring', 4: 'Spring', 5: 'Spring',
         6: 'Summer', 7: 'Summer', 8: 'Summer',
         9: 'Fall',  10: 'Fall',  11: 'Fall'
    })

    # Clean ZIPs and boroughs
    df_new['Incident Zip'] = df_new['Incident Zip'].astype(str).str.strip().str[:5]
    df_new['Borough']      = df_new['Borough'].str.upper()
    df_new = df_new[df_new['Borough'] != 'UNSPECIFIED'].copy()

    # Numeric lat/lon 
    df_new['Latitude']  = pd.to_numeric(df_new['Latitude'],  errors='coerce')
    df_new['Longitude'] = pd.to_numeric(df_new['Longitude'], errors='coerce')

    # Merge neighborhood lookup

    """
    This section is used to merge the neigbhorhood data with
    the 311 data to map zipcodes to neighborhoods.
    """

    neighborhoods = pd.read_csv(NEIGHBORHOODS_CSV)
    neighborhoods['ZipCode'] = neighborhoods['ZipCode'].astype(str).str.strip().str[:5]
    neighborhoods['Borough'] = neighborhoods['Borough'].str.upper()

     # Rename neighborhood borough BEFORE merge
    neighborhoods_renamed = neighborhoods.rename(columns={'Borough': 'Borough_Zip'})


    ## Preserve original 311 borough
    df_new['Borough_311'] = df_new['Borough']

   
    df_new = df_new.merge(
        neighborhoods_renamed[['ZipCode', 'Neighborhood', 'Borough_Zip']],
        left_on='Incident Zip',
        right_on='ZipCode',
        how='left'
    )

    # Prefer ZIP-based borough when available
    df_new['Borough'] = df_new['Borough_Zip'].fillna(df_new['Borough_311'])
    df_new.drop(columns=['ZipCode', 'Borough_311', 'Borough_Zip'], inplace=True, errors='ignore')

    print(f"Cleaned records: {len(df_new):,}")
    print(f"Neighborhood mapped: {df_new['Neighborhood'].notna().mean():.1%}")
    print(f"Date range: {df_new['Created Date'].min().date()} → {df_new['Created Date'].max().date()}")

Cleaned records: 4,195
Neighborhood mapped: 99.2%
Date range: 2026-05-31 → 2026-06-01


## Step 4: Deduplicate & Append

Load the existing parquet, drop any rows whose `Unique Key` already exists
(guards against re-fetching overlapping windows), then append and save.

In [6]:
if new_raw.empty:
    print("No new data — nothing to append.")

else:
    if Path(CLEANED_PARQUET).exists():
        df_existing = pd.read_parquet(CLEANED_PARQUET)

        print(f"Existing records: {len(df_existing):,}")

       
        if 'Unique Key' in df_existing.columns:
            df_existing['Unique Key'] = df_existing['Unique Key'].astype(str)

        if 'Unique Key' in df_new.columns:
            df_new['Unique Key'] = df_new['Unique Key'].astype(str)

        # Deduplicate on Unique Key
        if 'Unique Key' in df_existing.columns and 'Unique Key' in df_new.columns:

            existing_keys = set(df_existing['Unique Key'])

            before = len(df_new)

            df_new = df_new[
                ~df_new['Unique Key'].isin(existing_keys)
            ]

            dupes = before - len(df_new)

            if dupes:
                print(f"Dropped {dupes:,} duplicate records")

        # Align columns
        for col in df_existing.columns:
            if col not in df_new.columns:
                df_new[col] = np.nan

        df_new = df_new[df_existing.columns]

        df_combined = pd.concat(
            [df_existing, df_new],
            ignore_index=True
        )

    else:
        print("No existing parquet found — saving new data as initial dataset.")

        # Keep schema consistent from the start
        if 'Unique Key' in df_new.columns:
            df_new['Unique Key'] = df_new['Unique Key'].astype(str)

        df_combined = df_new

    # Optional safety step
    df_combined['Unique Key'] = df_combined['Unique Key'].astype(str)

    df_combined.to_parquet(
        CLEANED_PARQUET,
        index=False
    )

    print(f"\n✓ Appended {len(df_new):,} new records")
    print(f"✓ Total records now: {len(df_combined):,}")
    print(f"✓ Saved to: {CLEANED_PARQUET}")

Existing records: 4,865,731

✓ Appended 4,195 new records
✓ Total records now: 4,869,926
✓ Saved to: ../data/processed/noise_cleaned.parquet


## Step 5: Update Ingestion Timestamp

Write the timestamp of the latest fetched record so next run only pulls what's new.

In [7]:
if not new_raw.empty:
    latest_record = df_new['Created Date'].max()
    new_cutoff    = latest_record.strftime("%Y-%m-%dT%H:%M:%S")

    with open(LAST_INGESTED_FILE, 'w') as f:
        f.write(new_cutoff)

    print(f"✓ last_ingested.txt updated to: {new_cutoff}")
    print(f"  Next run will fetch records after this timestamp.")
else:
    print("No new data fetched — last_ingested.txt unchanged.")

✓ last_ingested.txt updated to: 2026-06-01T01:51:15
  Next run will fetch records after this timestamp.


## Step 6: Summary

In [8]:
print("=" * 50)
print("INGESTION COMPLETE")
print("=" * 50)

if not new_raw.empty:
    print(f"  New records added:   {len(df_new):,}")
    print(f"  Total in dataset:    {len(df_combined):,}")
    print(f"  Latest record date:  {new_cutoff}")
else:
    print("  Dataset already up to date.")

print()
print("Next steps to retrain on fresh data:")
print("  1. Re-run 01_data_loading.ipynb    (skip if no schema changes)")
print("  2. Re-run 02_feature_engineering.ipynb")
print("  3. Re-run 03_model_training.ipynb")

INGESTION COMPLETE
  New records added:   4,195
  Total in dataset:    4,869,926
  Latest record date:  2026-06-01T01:51:15

Next steps to retrain on fresh data:
  1. Re-run 01_data_loading.ipynb    (skip if no schema changes)
  2. Re-run 02_feature_engineering.ipynb
  3. Re-run 03_model_training.ipynb
